# Clasificación de estadios del sueño — Sleep-EDF Telemetry (v2)

Reemplazo directo de `model_jj.ipynb`. Cambios respecto a la versión anterior:

| # | Cambio | Por qué |
|---|--------|---------|
| 1 | **Agrupación por sujeto, no por registro** | En Telemetry `ST7[ss][n]J0`: `ss`=sujeto, `n`=noche. Las 2 noches de una persona iban a folds distintos → fuga de información |
| 2 | Descubrimiento automático de los 44 registros | Ya no hay lista manual de archivos |
| 3 | Recorte de vigilia (30 min antes/después del sueño) | Los registros traen horas de W en los extremos que desbalancean e inflan el accuracy |
| 4 | Normalización robusta **por registro** | La amplitud del EEG varía mucho entre personas; es la pieza clave para generalizar cross-subject |
| 5 | Contexto t−2…t+2 + media móvil de 11 épocas | `n_vecinos=1` era poco para la estructura temporal del sueño |
| 6 | Features nuevas: ondas lentas >75 µV, ráfagas tipo huso, ratios inter-canal, posición en la noche | Proxies directos de los criterios AASM de N3 y N2 |
| 7 | Entropía / frecuencia mediana / SEF95 restringidas a 0.5–30 Hz | Antes se calculaban hasta Nyquist (50 Hz) incluyendo la banda ya filtrada |
| 8 | LightGBM (con *fallback* a RandomForest) | Suele superar a RF en este problema tabular y entrena más rápido |
| 9 | Suavizado HMM + Viterbi sobre las predicciones | El sueño no salta de N3 a W; la matriz de transición recupera esa estructura |
| 10 | κ por fold en MLflow + comparación crudo vs. suavizado | Permite medir la variabilidad entre sujetos y aislar la ganancia de cada pieza |

> Todos los cambios están detrás de flags en la celda de configuración, para que puedas hacer ablaciones y reportar de dónde viene cada punto de mejora.

## Dependencias

```bash
pip install lightgbm
```

In [1]:
%pip install --upgrade mne
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 17.5 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 14.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [45]:
"""
=========================================================
 CLASIFICACION DE ETAPAS DE SUENO - Sleep-EDF Telemetry
 Canales: EEG Fpz-Cz, EEG Pz-Oz, EOG Horizontal, EMG Submental, 
 Pipeline: EDF -> recorte W -> epocas 30s -> features
           -> normalizacion por registro -> contexto
           -> LightGBM -> suavizado HMM (Viterbi)
=========================================================
"""

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.integrate import trapezoid   # np.trapz no existe en NumPy 2.x y
                                        # np.trapezoid no existe en NumPy 1.x;
                                        # este nombre funciona en ambos

import mne
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    cohen_kappa_score, f1_score, classification_report, confusion_matrix
)

try:
    from lightgbm import LGBMClassifier
    HAY_LGBM = True
except ImportError:
    HAY_LGBM = False
    warnings.warn("LightGBM no disponible; se usara RandomForest. pip install lightgbm")

mne.set_log_level("ERROR")

## Configuración

Las opciones del bloque *pipeline* son las que conviene ablacionar una por una
(`RECORTAR_W_MIN=None`, `NORMALIZAR_POR_REGISTRO=False`, `N_VECINOS=1`,
`USAR_SUAVIZADO_HMM=False`) para cuantificar el aporte de cada mejora en el informe.

In [46]:
# ── CONFIGURACION ────────────────────────────────────────
BASE = Path("../data/sleep-telemetry")
CANALES = ["EEG Fpz-Cz", "EEG Pz-Oz"]
EPOCA_SEG = 30.0
L_FREQ, H_FREQ = 0.3, 35.0
FUNDIR_N3_N4 = True

# Bandas para potencia espectral (Hz)
BANDAS = {
    "sw":    (0.5, 2.0),     # oscilacion lenta (criterio N3)
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "sigma": (12.0, 16.0),   # husos de sueno
    "beta":  (16.0, 30.0),
}
BANDA_UTIL = (0.5, 30.0)     # rango para entropia / freq mediana / SEF95

# Bandas que ademas se analizan en el dominio del tiempo
BANDA_SW = (0.5, 2.0)        # ondas lentas
BANDA_HUSO = (11.0, 16.0)    # husos

# ── OPCIONES DEL PIPELINE (para ablaciones) ──────────────
RECORTAR_W_MIN = 30          # min de vigilia a conservar antes/despues del sueno
NORMALIZAR_POR_REGISTRO = True
N_VECINOS = 2                # contexto t-2..t+2
VENTANAS_ROLLING = (11,)     # medias moviles adicionales (en epocas)
USAR_SUAVIZADO_HMM = True
PESO_CLASES = "balanced"     # None para desactivar

MAPA_ETAPAS = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N4",
    "Sleep stage R": "REM",
}
ETAPAS_VALIDAS = ["W", "N1", "N2", "N3", "REM"] if FUNDIR_N3_N4 \
    else ["W", "N1", "N2", "N3", "N4", "REM"]

## 0. Descubrimiento de archivos

**Este es el arreglo más importante.** En Sleep Telemetry cada persona durmió dos
noches (una con placebo y otra con temazepam), así que 22 sujetos = 44 registros.
El nombre `ST7011J0` significa sujeto 01, noche 1; `ST7021`/`ST7022` son las **dos
noches del sujeto 02**, no dos personas distintas.

Usar el nombre del registro como grupo hacía que las dos noches de la misma persona
pudieran caer en folds distintos — exactamente la fuga de información que la
propuesta pide evitar.

> **Nota:** el patrón se compara contra `psg.name`, **no** contra `psg.stem`.
> `Path("ST7011J0-PSG.edf").stem` devuelve `"ST7011J0-PSG"` (Python solo quita el
> último sufijo), así que un regex anclado en `J0$` no haría match nunca.

In [47]:
# =========================================================
# 0. DESCUBRIMIENTO DE ARCHIVOS  (agrupa las 2 noches por sujeto)
# =========================================================
# Sleep Telemetry: ST7[ss][n]J0-PSG.edf
#   ss = numero de sujeto (01..22),  n = noche (1 o 2)
# El hipnograma es ST7[ss][n]J?-Hypnogram.edf, donde ? es la
# inicial del tecnico que lo anoto (M o P) -> hay que hacer glob.
RE_ST = re.compile(r"^ST7(\d{2})(\d)J0-PSG\.edf$", re.IGNORECASE)


def descubrir_registros(base=BASE):
    """Escanea la carpeta y devuelve un registro por PSG con su sujeto y noche."""
    base = Path(base)
    if not base.exists():
        raise FileNotFoundError(f"No existe la carpeta {base.resolve()}")

    psgs = sorted(base.glob("*-PSG.edf"))
    if not psgs:
        muestra = sorted(p.name for p in base.iterdir())[:10]
        raise FileNotFoundError(
            f"No hay archivos *-PSG.edf en {base.resolve()}.\n"
            f"Contenido (primeros 10): {muestra}"
        )

    registros, omitidos = [], []
    for psg in psgs:
        # OJO: se matchea contra .name, no contra .stem. Path("ST7011J0-PSG.edf").stem
        # devuelve "ST7011J0-PSG" (solo quita el ultimo sufijo), no "ST7011J0".
        m = RE_ST.match(psg.name)
        if m is None:
            omitidos.append(psg.name)
            continue

        ss, noche = m.group(1), m.group(2)
        candidatos = sorted(base.glob(f"ST7{ss}{noche}J?-Hypnogram.edf"))
        if not candidatos:
            omitidos.append(f"{psg.name} (sin hipnograma)")
            continue

        registros.append({
            "psg": psg,
            "hyp": candidatos[0],
            "registro_id": f"ST7{ss}{noche}",
            "sujeto_id": f"S{ss}",        # <- CLAVE: las 2 noches comparten sujeto
            "noche": int(noche),
        })

    if omitidos:
        print(f"  [aviso] {len(omitidos)} archivos omitidos: {omitidos[:5]}"
              + (" ..." if len(omitidos) > 5 else ""))

    if not registros:
        raise RuntimeError(
            f"Se encontraron {len(psgs)} archivos PSG pero ninguno con nombre valido "
            f"de Sleep Telemetry (ST7ssnJ0-PSG.edf). Ejemplo hallado: {psgs[0].name}"
        )

    n_suj = len({r["sujeto_id"] for r in registros})
    print(f"{len(registros)} registros de {n_suj} sujetos distintos")
    return registros

## 1. Carga, recorte de vigilia y segmentación

Las copias filtradas por banda (`raw_sw`, `raw_huso`) se calculan **una vez sobre la
señal continua**, no época por época: es mucho más rápido y evita artefactos de borde
del filtro en cada segmento de 30 s.

In [48]:
# =========================================================
# 1. CARGA, RECORTE DE VIGILIA Y SEGMENTACION
# =========================================================
def cargar_registro(psg_file, hyp_file, canales=CANALES):
    """Carga el PSG, filtra, y prepara las copias filtradas por banda."""
    raw = mne.io.read_raw_edf(psg_file, preload=True, verbose="ERROR")

    disponibles = [c for c in canales if c in raw.ch_names]
    if not disponibles:
        raise ValueError(f"Ningun canal de {canales} en {psg_file.name}. "
                         f"Disponibles: {raw.ch_names}")
    raw.pick(disponibles)

    # Pasa-banda general: quita deriva DC y ruido de alta frecuencia
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)

    # Copias filtradas para analisis temporal (ondas lentas y husos).
    # Se filtra la senal CONTINUA una sola vez, no epoca por epoca:
    # es mas rapido y evita artefactos de borde.
    raw_sw = raw.copy().filter(*BANDA_SW, verbose=False)
    raw_huso = raw.copy().filter(*BANDA_HUSO, verbose=False)

    ann = mne.read_annotations(hyp_file)
    return raw, raw_sw, raw_huso, ann, disponibles


def etiquetas_por_epoca(ann, n_epocas, epoca_seg=EPOCA_SEG):
    """Anotaciones -> vector de etiquetas, una por epoca (None si no hay)."""
    etiquetas = np.full(n_epocas, None, dtype=object)

    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        etapa = MAPA_ETAPAS.get(str(desc).strip())
        if etapa is None:
            continue                      # "Movement time", "Sleep stage ?"
        if FUNDIR_N3_N4 and etapa == "N4":
            etapa = "N3"

        i_ini = int(np.floor(onset / epoca_seg))
        i_fin = int(np.ceil((onset + dur) / epoca_seg))
        i_ini, i_fin = max(0, i_ini), min(n_epocas, i_fin)
        etiquetas[i_ini:i_fin] = etapa

    return etiquetas


def segmentar(datos, fs, epoca_seg=EPOCA_SEG):
    """(n_canales, n_total) -> (n_epocas, n_canales, n_muestras)."""
    n_canales, n_total = datos.shape
    muestras_epoca = int(round(epoca_seg * fs))
    n_epocas = n_total // muestras_epoca

    datos = datos[:, :n_epocas * muestras_epoca]
    epocas = datos.reshape(n_canales, n_epocas, muestras_epoca)
    return np.transpose(epocas, (1, 0, 2))


def indices_recorte(y, minutos=RECORTAR_W_MIN, epoca_seg=EPOCA_SEG):
    """
    Conserva solo `minutos` de vigilia antes del primer estadio de sueno
    y despues del ultimo. Los registros de Sleep-EDF traen horas de W en
    los extremos que desbalancean el dataset e inflan el accuracy.
    """
    if minutos is None:
        return 0, len(y)

    n_margen = int(minutos * 60 / epoca_seg)
    es_sueno = np.array([e in ("N1", "N2", "N3", "REM") for e in y])
    if not es_sueno.any():
        return 0, len(y)

    primero = int(np.argmax(es_sueno))
    ultimo = len(y) - 1 - int(np.argmax(es_sueno[::-1]))
    return max(0, primero - n_margen), min(len(y), ultimo + 1 + n_margen)

## 2. Extracción de features

Novedades frente a la versión anterior:

- **`sw_frac75`** — fracción de la época con ondas 0.5–2 Hz de amplitud >75 µV pico a pico.
  Es literalmente el criterio AASM para N3.
- **`huso_n_rafagas`** — número de tramos de 0.5–2 s donde la envolvente en 11–16 Hz
  supera 2× su mediana. Proxy de husos de sueño, marcador clave de N2.
- **`inter_ratio_*`** — ratios Fpz-Cz / Pz-Oz. El alfa occipital distingue vigilia relajada.
- **`pos_relativa`** — posición en la noche. N3 se concentra al principio, REM se acumula al final.
- **`average="median"`** en Welch, más robusto a artefactos que la media.
- Entropía, frecuencia mediana y SEF95 restringidas a 0.5–30 Hz.

> **Compatibilidad NumPy:** la integración de la PSD usa `scipy.integrate.trapezoid`.
> `np.trapz` fue eliminado en NumPy 2.x y `np.trapezoid` no existe en NumPy 1.x,
> así que ninguno de los dos nombres funciona en ambas versiones; el de scipy sí.

In [49]:
# =========================================================
# 2. EXTRACCION DE FEATURES
# =========================================================
def parametros_hjorth(x):
    """Actividad, movilidad y complejidad de Hjorth."""
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x, var_dx, var_ddx = np.var(x), np.var(dx), np.var(ddx)

    movilidad = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    mov_dx = np.sqrt(var_ddx / var_dx) if var_dx > 0 else 0.0
    complejidad = mov_dx / movilidad if movilidad > 0 else 0.0
    return var_x, movilidad, complejidad


def features_canal(x, x_sw, x_huso, fs, prefijo):
    """Features de un canal para una epoca. x_sw/x_huso vienen prefiltrados."""
    f = {}
    eps = 1e-12

    # --- Espectro via Welch (ventanas de 4 s -> resolucion 0.25 Hz) -----
    # average="median" es mas robusto a artefactos puntuales que la media.
    nperseg = min(len(x), int(4 * fs))
    freqs, psd = signal.welch(x, fs=fs, nperseg=nperseg, average="median")

    potencias = {}
    for nombre, (lo, hi) in BANDAS.items():
        mask = (freqs >= lo) & (freqs < hi)
        potencias[nombre] = float(trapezoid(psd[mask], freqs[mask])) if mask.any() else 0.0

    # "delta" contiene a "sw"; el total se calcula sin sw para no contar doble
    total = sum(p for n, p in potencias.items() if n != "sw")

    for nombre, p in potencias.items():
        f[f"{prefijo}_rel_{nombre}"] = p / total if total > 0 else 0.0
        f[f"{prefijo}_log_{nombre}"] = float(np.log10(p + eps))

    # Ratios entre bandas: muy discriminativos entre etapas
    f[f"{prefijo}_ratio_delta_beta"] = potencias["delta"] / (potencias["beta"] + eps)
    f[f"{prefijo}_ratio_theta_alpha"] = potencias["theta"] / (potencias["alpha"] + eps)
    f[f"{prefijo}_ratio_delta_theta"] = potencias["delta"] / (potencias["theta"] + eps)
    f[f"{prefijo}_ratio_alpha_beta"] = potencias["alpha"] / (potencias["beta"] + eps)
    f[f"{prefijo}_ratio_sigma_theta"] = potencias["sigma"] / (potencias["theta"] + eps)

    # --- Forma del espectro, restringida a la banda util -----------------
    # (antes se calculaba hasta Nyquist=50 Hz, pero la senal esta filtrada
    #  a 35 Hz: esa cola vacia distorsionaba sobre todo el SEF95)
    util = (freqs >= BANDA_UTIL[0]) & (freqs <= BANDA_UTIL[1])
    freqs_u, psd_u = freqs[util], psd[util]

    psd_norm = psd_u / (psd_u.sum() + eps)
    f[f"{prefijo}_entropia_espectral"] = float(-np.sum(psd_norm * np.log2(psd_norm + eps)))

    acum = np.cumsum(psd_u)
    if acum[-1] > 0:
        acum = acum / acum[-1]
        f[f"{prefijo}_freq_mediana"] = float(freqs_u[np.searchsorted(acum, 0.50)])
        f[f"{prefijo}_sef95"] = float(freqs_u[np.searchsorted(acum, 0.95)])
    else:
        f[f"{prefijo}_freq_mediana"] = 0.0
        f[f"{prefijo}_sef95"] = 0.0

    # --- Estadisticas en el dominio del tiempo ---------------------------
    f[f"{prefijo}_std"] = float(np.std(x))
    f[f"{prefijo}_ptp"] = float(x.max() - x.min())
    f[f"{prefijo}_kurtosis"] = float(stats.kurtosis(x))
    f[f"{prefijo}_skew"] = float(stats.skew(x))
    f[f"{prefijo}_p75_abs"] = float(np.percentile(np.abs(x), 75))
    f[f"{prefijo}_zcr"] = float(np.mean(np.diff(np.signbit(x)) != 0))

    act, mov, comp = parametros_hjorth(x)
    f[f"{prefijo}_hjorth_actividad"] = float(act)
    f[f"{prefijo}_hjorth_movilidad"] = float(mov)
    f[f"{prefijo}_hjorth_complejidad"] = float(comp)

    # --- Ondas lentas: proxy directo del criterio AASM de N3 -------------
    # N3 = >=20% de la epoca con ondas 0.5-2 Hz de amplitud pico-pico >75 uV
    env_sw = np.abs(signal.hilbert(x_sw))
    f[f"{prefijo}_sw_frac75"] = float(np.mean(env_sw > 37.5))   # 75 uV pp / 2
    f[f"{prefijo}_sw_frac40"] = float(np.mean(env_sw > 20.0))
    f[f"{prefijo}_sw_env_p90"] = float(np.percentile(env_sw, 90))
    f[f"{prefijo}_sw_rms"] = float(np.sqrt(np.mean(x_sw ** 2)))

    # --- Husos de sueno: marcador clave de N2 ----------------------------
    env_h = np.abs(signal.hilbert(x_huso))
    med_h = np.median(env_h) + eps
    f[f"{prefijo}_huso_rms"] = float(np.sqrt(np.mean(x_huso ** 2)))
    f[f"{prefijo}_huso_env_p90"] = float(np.percentile(env_h, 90))
    f[f"{prefijo}_huso_pico_med"] = float(np.max(env_h) / med_h)

    # Rafagas tipo huso: tramos donde la envolvente supera 2x su mediana
    # durante 0.5-2 s (duracion tipica de un huso)
    sobre = env_h > 2.0 * med_h
    bordes = np.diff(np.concatenate(([0], sobre.astype(int), [0])))
    inicios, finales = np.where(bordes == 1)[0], np.where(bordes == -1)[0]
    duraciones = (finales - inicios) / fs
    f[f"{prefijo}_huso_n_rafagas"] = float(np.sum((duraciones >= 0.5) & (duraciones <= 2.0)))

    return f


def extraer_features(epocas, epocas_sw, epocas_huso, fs, nombres_canales):
    """DataFrame de features, una fila por epoca."""
    filas = []
    n_epocas = epocas.shape[0]

    for i in range(n_epocas):
        fila = {}
        for c, nombre in enumerate(nombres_canales):
            prefijo = nombre.replace("EEG ", "").replace("-", "")
            fila.update(features_canal(
                epocas[i, c, :], epocas_sw[i, c, :], epocas_huso[i, c, :], fs, prefijo
            ))

        # Posicion relativa en la noche: N3 se concentra al principio,
        # REM se acumula hacia el final. Feature barata y util.
        fila["pos_relativa"] = i / max(1, n_epocas - 1)
        filas.append(fila)

    X = pd.DataFrame(filas)

    # Ratios entre canales (alfa occipital distingue vigilia relajada)
    if len(nombres_canales) == 2:
        p0 = nombres_canales[0].replace("EEG ", "").replace("-", "")
        p1 = nombres_canales[1].replace("EEG ", "").replace("-", "")
        eps = 1e-12
        for banda in ("delta", "theta", "alpha", "sigma", "beta"):
            X[f"inter_ratio_{banda}"] = (
                X[f"{p0}_rel_{banda}"] / (X[f"{p1}_rel_{banda}"] + eps)
            )
        X["inter_ratio_std"] = X[f"{p0}_std"] / (X[f"{p1}_std"] + eps)

    return X

## 3. Normalización por registro y contexto temporal

La normalización se aplica **antes** del contexto y **dentro** de cada registro. No hay
fuga de información: no usa etiquetas ni datos de otros sujetos, solo reescala cada
grabación contra su propia distribución.

In [50]:
# =========================================================
# 3. NORMALIZACION POR REGISTRO Y CONTEXTO TEMPORAL
# =========================================================
def normalizar_por_registro(X):
    """
    Z-score robusto (mediana / IQR) DENTRO de cada registro.

    Es la pieza mas importante para generalizar entre sujetos: la amplitud
    del EEG varia enormemente entre personas (impedancia, grosor del craneo,
    montaje), asi que features absolutas como log_delta, std o ptp no son
    comparables tal cual. Al calcularse por registro no hay fuga de
    informacion: no usa etiquetas ni datos de otros sujetos.
    """
    mediana = X.median()
    iqr = (X.quantile(0.75) - X.quantile(0.25)).replace(0, np.nan).fillna(1.0)
    Xn = (X - mediana) / iqr
    return Xn.replace([np.inf, -np.inf], 0.0).fillna(0.0)


def agregar_contexto(X, n_vecinos=N_VECINOS, ventanas=VENTANAS_ROLLING):
    """
    Concatena las features de las epocas vecinas (t-n ... t+n) y medias
    moviles centradas. Las etapas de sueno tienen fuerte estructura
    temporal: un contexto mas ancho que +-1 ayuda bastante.
    """
    partes = [X.add_suffix("_t0")]

    for k in range(1, n_vecinos + 1):
        partes.append(X.shift(k).add_suffix(f"_t-{k}"))
        partes.append(X.shift(-k).add_suffix(f"_t+{k}"))

    for w in ventanas:
        partes.append(
            X.rolling(w, center=True, min_periods=1).mean().add_suffix(f"_movil{w}")
        )

    X_ctx = pd.concat(partes, axis=1)
    return X_ctx.bfill().ffill()

## 4. Pipeline por registro

Orden importante: recortar → extraer features → normalizar → contexto → descartar
épocas inválidas. El contexto debe calcularse sobre épocas **contiguas**, antes de
eliminar las `?` y `Movement time`, para no emparejar vecinos que no lo son.

In [51]:
# =========================================================
# 4. PIPELINE POR REGISTRO
# =========================================================
def procesar_registro(reg, n_vecinos=N_VECINOS):
    """Devuelve (X, y, meta) de un registro, listo para el modelo."""
    raw, raw_sw, raw_huso, ann, canales = cargar_registro(reg["psg"], reg["hyp"])
    fs = raw.info["sfreq"]

    epocas = segmentar(raw.get_data() * 1e6, fs)
    epocas_sw = segmentar(raw_sw.get_data() * 1e6, fs)
    epocas_huso = segmentar(raw_huso.get_data() * 1e6, fs)

    y = etiquetas_por_epoca(ann, n_epocas=epocas.shape[0])

    # Recorte de vigilia ANTES de extraer features (ahorra mucho computo)
    i0, i1 = indices_recorte(y)
    epocas, epocas_sw, epocas_huso = epocas[i0:i1], epocas_sw[i0:i1], epocas_huso[i0:i1]
    y = y[i0:i1]

    X = extraer_features(epocas, epocas_sw, epocas_huso, fs, canales)

    if NORMALIZAR_POR_REGISTRO:
        X = normalizar_por_registro(X)

    # El contexto se calcula sobre epocas contiguas, ANTES de descartar
    # las invalidas, para no mezclar vecinos que no lo son.
    X = agregar_contexto(X, n_vecinos=n_vecinos)

    valido = np.array([e in ETAPAS_VALIDAS for e in y])
    X, y = X[valido].reset_index(drop=True), y[valido]

    meta = pd.DataFrame({
        "sujeto_id": reg["sujeto_id"],
        "registro_id": reg["registro_id"],
        "noche": reg["noche"],
    }, index=range(len(y)))

    print(f"  {reg['registro_id']} (suj {reg['sujeto_id']}, n{reg['noche']}): "
          f"{len(y)} epocas validas [recorte {i0}:{i1}]")

    return X, pd.Series(y, name="etapa"), meta


def construir_dataset(registros, n_vecinos=N_VECINOS):
    """Concatena todos los registros."""
    Xs, ys, ms = [], [], []
    for reg in registros:
        try:
            X, y, m = procesar_registro(reg, n_vecinos=n_vecinos)
        except Exception as e:                      # noqa: BLE001
            print(f"  [error] {reg['registro_id']}: {e}")
            continue
        Xs.append(X)
        ys.append(y)
        ms.append(m)

    if not Xs:
        raise RuntimeError(
            f"Ninguno de los {len(registros)} registros pudo procesarse. "
            "Revisa los mensajes [error] de arriba."
        )

    X = pd.concat(Xs, ignore_index=True)
    y = pd.concat(ys, ignore_index=True)
    meta = pd.concat(ms, ignore_index=True)
    return X, y, meta

## 5. Suavizado temporal con HMM (Viterbi)

La matriz de transición se estima **solo con los hipnogramas del fold de entrenamiento**.
La posterior del modelo se convierte en verosimilitud dividiendo por el prior de clase,
y Viterbi se corre **por registro** (no sobre la concatenación de todos).

Es de las mejoras con mejor relación coste/beneficio: típicamente 2–4 puntos de κ.

In [52]:
# =========================================================
# 5. SUAVIZADO TEMPORAL CON HMM (Viterbi)
# =========================================================
def matriz_transicion(secuencias, clases, suavizado=1.0):
    """Estima P(estado_t | estado_t-1) y la distribucion inicial."""
    idx = {c: i for i, c in enumerate(clases)}
    K = len(clases)
    A = np.full((K, K), suavizado)
    pi = np.full(K, suavizado)

    for seq in secuencias:
        seq = list(seq)
        if not seq:
            continue
        pi[idx[seq[0]]] += 1
        for a, b in zip(seq[:-1], seq[1:]):
            A[idx[a], idx[b]] += 1

    return A / A.sum(axis=1, keepdims=True), pi / pi.sum()


def viterbi(log_emision, A, pi):
    """Camino de estados mas probable dado log P(x_t | estado)."""
    n, K = log_emision.shape
    logA, logpi = np.log(A + 1e-12), np.log(pi + 1e-12)

    delta = np.zeros((n, K))
    psi = np.zeros((n, K), dtype=int)
    delta[0] = logpi + log_emision[0]

    for t in range(1, n):
        m = delta[t - 1][:, None] + logA
        psi[t] = np.argmax(m, axis=0)
        delta[t] = m[psi[t], np.arange(K)] + log_emision[t]

    camino = np.zeros(n, dtype=int)
    camino[-1] = int(np.argmax(delta[-1]))
    for t in range(n - 2, -1, -1):
        camino[t] = psi[t + 1, camino[t + 1]]
    return camino


def suavizar_hmm(proba, clases, A, pi, prior, registros_epoca):
    """
    Aplica Viterbi por registro. Convierte la posterior del modelo en
    verosimilitud dividiendo por el prior de clase:
        log P(x|y) = log P(y|x) - log P(y) + const
    """
    pred = np.empty(len(proba), dtype=object)
    log_emision_total = np.log(proba + 1e-12) - np.log(prior + 1e-12)

    for rid in pd.unique(registros_epoca):
        m = (registros_epoca == rid).to_numpy()
        camino = viterbi(log_emision_total[m], A, pi)
        pred[m] = [clases[i] for i in camino]

    return pred

## 6. Evaluación y modelo

In [53]:
# =========================================================
# 6. EVALUACION
# =========================================================
def evaluar(y_true, y_pred, etiquetas=None, prefijo="", verbose=True):
    """Metricas estandar en sleep staging."""
    etiquetas = etiquetas or [e for e in ETAPAS_VALIDAS if e in set(y_true)]

    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    acc = (y_true == y_pred).mean()
    kappa = cohen_kappa_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_por_clase = f1_score(y_true, y_pred, average=None,
                            labels=etiquetas, zero_division=0)

    if verbose:
        print(f"\n  Accuracy      : {acc:.4f}")
        print(f"  Cohen's kappa : {kappa:.4f}   <- metrica de referencia")
        print(f"  F1 macro      : {f1:.4f}\n")
        print(classification_report(y_true, y_pred, zero_division=0))

        cm = pd.DataFrame(
            confusion_matrix(y_true, y_pred, labels=etiquetas),
            index=[f"real_{e}" for e in etiquetas],
            columns=[f"pred_{e}" for e in etiquetas],
        )
        print("Matriz de confusion:")
        print(cm)
        print("\nMatriz normalizada por fila (recall por estadio):")
        print((cm.div(cm.sum(axis=1), axis=0)).round(3))

    if mlflow.active_run() is not None:
        mlflow.log_metric(f"{prefijo}accuracy", float(acc))
        mlflow.log_metric(f"{prefijo}kappa", float(kappa))
        mlflow.log_metric(f"{prefijo}f1_macro", float(f1))
        for e, v in zip(etiquetas, f1_por_clase):
            mlflow.log_metric(f"{prefijo}f1_{e}", float(v))

    return dict(accuracy=acc, kappa=kappa, f1_macro=f1,
                f1_por_clase=dict(zip(etiquetas, f1_por_clase)))


def crear_modelo(semilla=42):
    """LightGBM si esta disponible; si no, RandomForest equivalente."""
    if HAY_LGBM:
        return LGBMClassifier(
            objective="multiclass",
            n_estimators=700,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=40,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.6,
            reg_lambda=1.0,
            class_weight=PESO_CLASES,
            n_jobs=-1,
            random_state=semilla,
            verbose=-1,
        )
    return RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample" if PESO_CLASES else None,
        n_jobs=-1,
        random_state=semilla,
    )

## 7. Entrenamiento con validación cruzada por sujeto

`n_splits=5` da una CV rápida para iterar. Para el resultado final del informe,
pon **`n_splits=22`** y tendrás LOSO estricto, que es lo que pide la propuesta.

El bucle reporta κ crudo y κ con suavizado en cada fold, así que puedes cuantificar
exactamente cuánto aporta el HMM.

In [54]:
# =========================================================
# 7. ENTRENAMIENTO CON VALIDACION CRUZADA POR SUJETO
# =========================================================
def entrenar_cv(X, y, meta, n_splits=5, semilla=42):
    """
    GroupKFold agrupando por SUJETO (no por registro): las dos noches de
    la misma persona van siempre al mismo fold. Con n_splits = n_sujetos
    se obtiene LOSO estricto.
    """
    grupos = meta["sujeto_id"].to_numpy()
    n_sujetos = len(np.unique(grupos))
    n_splits = min(n_splits, n_sujetos)
    print(f"\nGroupKFold: {n_splits} folds sobre {n_sujetos} sujetos "
          f"({len(np.unique(meta['registro_id']))} registros)")

    gkf = GroupKFold(n_splits=n_splits)
    clases = np.array(sorted(set(y)))

    y_true_all, y_pred_all, y_pred_hmm_all = [], [], []
    kappas_crudo, kappas_hmm = [], []

    for fold, (i_tr, i_te) in enumerate(gkf.split(X, y, groups=grupos), 1):
        modelo = crear_modelo(semilla)
        modelo.fit(X.iloc[i_tr], y.iloc[i_tr])

        proba = modelo.predict_proba(X.iloc[i_te])
        clases_m = np.asarray(modelo.classes_)
        pred = clases_m[np.argmax(proba, axis=1)]

        # HMM estimado SOLO con los hipnogramas de entrenamiento
        meta_tr = meta.iloc[i_tr]
        secuencias = [y.iloc[i_tr][meta_tr["registro_id"] == r].tolist()
                      for r in pd.unique(meta_tr["registro_id"])]
        A, pi = matriz_transicion(secuencias, list(clases_m))
        prior = np.array([(y.iloc[i_tr] == c).mean() for c in clases_m])

        pred_hmm = suavizar_hmm(
            proba, list(clases_m), A, pi, prior,
            meta.iloc[i_te]["registro_id"].reset_index(drop=True),
        ) if USAR_SUAVIZADO_HMM else pred

        k_c = cohen_kappa_score(y.iloc[i_te], pred)
        k_h = cohen_kappa_score(y.iloc[i_te], pred_hmm)
        kappas_crudo.append(k_c)
        kappas_hmm.append(k_h)
        print(f"  Fold {fold}: kappa crudo = {k_c:.4f} | con HMM = {k_h:.4f}")

        if mlflow.active_run() is not None:
            mlflow.log_metric("kappa_fold_crudo", float(k_c), step=fold)
            mlflow.log_metric("kappa_fold_hmm", float(k_h), step=fold)

        y_true_all.extend(y.iloc[i_te])
        y_pred_all.extend(pred)
        y_pred_hmm_all.extend(pred_hmm)

    print(f"\nkappa por fold  crudo: {np.mean(kappas_crudo):.4f} "
          f"+/- {np.std(kappas_crudo):.4f}")
    print(f"kappa por fold  HMM  : {np.mean(kappas_hmm):.4f} "
          f"+/- {np.std(kappas_hmm):.4f}")

    print("\n" + "=" * 60)
    print("SIN suavizado temporal")
    print("=" * 60)
    m_crudo = evaluar(y_true_all, y_pred_all, prefijo="crudo_")

    print("\n" + "=" * 60)
    print("CON suavizado HMM (Viterbi)")
    print("=" * 60)
    m_hmm = evaluar(y_true_all, y_pred_hmm_all, prefijo="hmm_")

    # Modelo final entrenado con todo
    modelo_final = crear_modelo(semilla)
    modelo_final.fit(X, y)

    secuencias = [y[meta["registro_id"] == r].tolist()
                  for r in pd.unique(meta["registro_id"])]
    A_final, pi_final = matriz_transicion(secuencias, list(modelo_final.classes_))
    prior_final = np.array([(y == c).mean() for c in modelo_final.classes_])

    return modelo_final, dict(crudo=m_crudo, hmm=m_hmm,
                              A=A_final, pi=pi_final, prior=prior_final,
                              kappas_crudo=kappas_crudo, kappas_hmm=kappas_hmm)


def importancia_features(modelo, X, top=25):
    imp = pd.Series(modelo.feature_importances_, index=X.columns)
    return imp.sort_values(ascending=False).head(top)

## 8. Ejecución

La desviación estándar de κ entre folds es tan informativa como la media: te dice
cuánto varía el rendimiento de sujeto a sujeto, que es justo el problema de
generalización cross-subject que plantea tu informe.

In [55]:
# =========================================================
# 8. EJECUCION
# =========================================================
mlflow.set_tracking_uri("http://100.27.188.254:5000")
experiment = mlflow.set_experiment("/model_jj")
if mlflow.active_run() is not None:
    mlflow.end_run()

registros = descubrir_registros(BASE)

with mlflow.start_run(experiment_id=experiment.experiment_id,
                      run_name="lgbm_ctx2_norm_hmm"):
    for k, v in {
        "modelo": "LightGBM" if HAY_LGBM else "RandomForest",
        "canales": CANALES,
        "EPOCA_SEG": EPOCA_SEG,
        "L_FREQ": L_FREQ,
        "H_FREQ": H_FREQ,
        "BANDAS": BANDAS,
        "recortar_W_min": RECORTAR_W_MIN,
        "normalizar_por_registro": NORMALIZAR_POR_REGISTRO,
        "n_vecinos": N_VECINOS,
        "ventanas_rolling": VENTANAS_ROLLING,
        "suavizado_hmm": USAR_SUAVIZADO_HMM,
        "peso_clases": PESO_CLASES,
        "n_registros": len(registros),
        "n_sujetos": len({r["sujeto_id"] for r in registros}),
    }.items():
        mlflow.log_param(k, v)

    print("\nConstruyendo dataset...")
    X, y, meta = construir_dataset(registros, n_vecinos=N_VECINOS)

    print(f"\nDataset: {X.shape[0]} epocas x {X.shape[1]} features")
    print("\nDistribucion de clases:")
    dist = y.value_counts().sort_index()
    print(pd.DataFrame({"n": dist, "%": (100 * dist / len(y)).round(2)}))

    mlflow.log_param("n_epocas", int(X.shape[0]))
    mlflow.log_param("n_features", int(X.shape[1]))

    modelo, res = entrenar_cv(X, y, meta, n_splits=5)

    print("\n--- Top 25 features mas importantes ---")
    print(importancia_features(modelo, X, top=25))

44 registros de 22 sujetos distintos

Construyendo dataset...
  ST7011 (suj S01, n1): 1092 epocas validas [recorte 0:1110]
  ST7012 (suj S01, n2): 1040 epocas validas [recorte 0:1044]
  ST7021 (suj S02, n1): 920 epocas validas [recorte 0:968]
  ST7022 (suj S02, n2): 944 epocas validas [recorte 0:984]
  ST7041 (suj S04, n1): 1007 epocas validas [recorte 0:1038]
  ST7042 (suj S04, n2): 1107 epocas validas [recorte 0:1126]
  ST7051 (suj S05, n1): 1018 epocas validas [recorte 0:1073]
  ST7052 (suj S05, n2): 1034 epocas validas [recorte 0:1091]
  ST7061 (suj S06, n1): 1008 epocas validas [recorte 0:1057]
  ST7062 (suj S06, n2): 952 epocas validas [recorte 0:998]
  ST7071 (suj S07, n1): 821 epocas validas [recorte 0:847]
  ST7072 (suj S07, n2): 803 epocas validas [recorte 0:804]
  ST7081 (suj S08, n1): 947 epocas validas [recorte 19:966]
  ST7082 (suj S08, n2): 931 epocas validas [recorte 0:951]
  ST7091 (suj S09, n1): 943 epocas validas [recorte 0:995]
  ST7092 (suj S09, n2): 923 epocas val

## Siguientes pasos sugeridos

1. **Ablaciones.** Corre el notebook desactivando una mejora a la vez y anota el κ.
   Es material directo para la sección de resultados del informe.
2. **LOSO estricto** (`n_splits=22`) para la cifra final comparable con la literatura.
3. **Ajuste de umbral para N1.** Es la clase más difícil (F1 típico 30–55%). Con
   `predict_proba` puedes subir su recall a costa de N2, y reportar la curva del
   compromiso.
4. **Análisis noche 1 vs. noche 2.** Tienes la columna `noche` en `meta`: la noche con
   temazepam altera la arquitectura del sueño (suprime N3, aumenta actividad sigma).
   Comparar el rendimiento entre ambas es un análisis de robustez propio y original,
   y te sustituye el benchmark cross-cohorte Cassette→Telemetry que no puedes hacer.
5. **Deep learning.** Con el pipeline de datos ya resuelto, TinySleepNet sobre la señal
   cruda es el siguiente paso natural — y el suavizado HMM de este notebook se le puede
   aplicar tal cual encima.

### Sobre las métricas esperadas

Con Telemetry (22 sujetos, población con dificultad para dormir) los números tienden a
ser algo **más bajos** que los publicados para Sleep-EDF-20/78, que usan Cassette con
sujetos sanos. Cuando compares contra el estado del arte, aclara el subconjunto: no es
comparable directo, y conviene decirlo explícitamente en el informe.

In [ ]:
print("Tracking URI:", mlflow.get_tracking_uri())
if mlflow.active_run() is not None:
    print("Artifact URI:", mlflow.get_artifact_uri())
else:
    print("Artifact URI: disponible durante un run activo")

2026/09/03 19:53:49 WARNING mlflow.tracking.fluent: No active run found. A new active run will be created. If this is not intended, please create a run using `mlflow.start_run()` first.


Tracking URI: sqlite:///C:/Users/ingju/OneDrive/2026/MAIA/Bimestre%202026-14/Desarrollo%20Proyectos%20IA/Proyectos/micro-proyecto-grupo-7/Notebooks/mlflow.db
Artifact URI: file:c:/Users/ingju/OneDrive/2026/MAIA/Bimestre 2026-14/Desarrollo Proyectos IA/Proyectos/micro-proyecto-grupo-7/Notebooks/mlruns/1/4ec64173f0f6418587c2ecca5da4d315/artifacts
